In a simulation competition you cannot evaluate everything. There are thousands of recorded games, judging one candidate properly costs a hundred matches, and a submission slot is a limited daily resource that answers hours later. So you need two different things: a **cheap filter** that decides what is worth evaluating at all, and an **honest judge** that decides what is worth submitting.

This notebook is about that machinery, not about a result it produced. It is the instrument's fourth generation, and most of what distinguishes it from the third is scar tissue: the sections below say plainly which parts are design and which parts exist because a specific measurement lied to me first.

Three of those lies are worth naming up front, because each one survived weeks of use and internal consistency checks:

* an offline corpus that correlated **0.848** with the leaderboard when validated, then predicted 72 percent for an agent that delivered 13;
* a rival panel built from the **strongest** public agents, whose ranking of five candidates came out in the **opposite order** to the same candidates measured against 344 recorded ladder routes;
* an evaluation whose default control was subtly not the agent being improved, so one arm measured exactly nothing, 0 discordant games in 816, while a differently-built census insisted the same change moved 640 turns a game. Both instruments were honest. The invocation asked the wrong question.

---

### What this notebook does not claim

It does **not** claim to have found the mechanism that makes an agent win, and it does **not** claim the quantities it filters on are that mechanism. Everything below is an instrument: a way of deciding what to measure next and how much to believe the measurement. An instrument can be useful and still be wrong about why it works.


### Related notebooks

- [Everyone is playing the same opening](https://www.kaggle.com/code/destbreso/everyone-is-playing-the-same-opening): the field is homogeneous, which is why a pool repeats itself.
- [How many opponents is a pool of forty?](https://www.kaggle.com/code/destbreso/how-many-opponents-is-a-pool-of-forty): what a pool is actually made of, and the correction it forces here.
- [Wins, not money](https://www.kaggle.com/code/destbreso/wins-not-money): what the judge optimises, derived and tested.
- [Kaggriculture Measure Your Agent](https://www.kaggle.com/code/destbreso/kaggriculture-measure-your-agent): the runnable harness the stages below feed.
- [Know Your Noise](https://www.kaggle.com/code/destbreso/kaggriculture-know-your-noise): how big the environment's own dice are, which every number here sits on top of.

Read together they are one argument: the field repeats itself, so any pool drawn from it repeats itself, so a screen counted in raw games overstates what it knows.


## 1. The problem, in terms of what things cost

Four costs, wildly asymmetric, and the whole design follows from their ratios.

| action | cost |
|---|---|
| download a recorded game | seconds |
| compare two recordings | free, no games played |
| evaluate a candidate against a pool | ~80 simulated matches |
| submit and find out | a limited daily slot, and hours of ladder time |

Because the last row is thousands of times more expensive than the first, the instrument must be a **funnel**: each stage cheap enough to run on everything that survived the stage before, and each stage's job clearly distinct from the next one's.

The failure mode this invites is subtle: it is very easy to build a funnel where every stage measures roughly the same thing. Then you have one instrument run four times, and four confirmations that mean one.

## 2. The shape we arrived at

```
  thousands of recorded streams
          |
  [1] FILTER    free       -> is this recording even transferable?
          |
  [1b] DEDUPE   free       -> is it a copy of something already here?
          |
  [2] SCREEN    ~100 games -> order the survivors, on a DECLARED population
          |
  [3] JUDGE     two questions, both in writing:
          |        Q1  does it beat the submission we already field,
          |            PAIRED and LIVE, on that submission's real matchups?
          |        Q2  is it competitive to field at all: does its macro
          |            BEHAVIOUR stand next to the leader's, before money?
          |
  [4] LIVE      one slot   -> and never read the score too early
```

Stages 1 and 2 answer different questions on purpose. Stage 1 asks whether a recording *means anything* away from the game it was recorded in. Stage 2 asks whether it is *good*, against a population that has to be named out loud, for reasons section 5 earned the hard way. Stage 3 stopped being one number in generation four: a candidate can beat the incumbent and still embarrass a slot, and a candidate can look magnificent in aggregate while every extra win comes from games that carry no information.

Step 1b is what keeps stages 2 and 3 from spending their budget comparing a candidate with something already in the set.


## 3. Stage 1, the filter, and the distinction that fixed it

Our first filter was binary: prove the source agent plays the *same actions regardless of the opponent*, then replay it. Anything that varied was rejected.

That filter rejected about 86 % of the field, which felt like rigour. It was actually a **conflation of two different requirements that happened to share a name**:

- As a property of a **sparring partner**, determinism is indispensable. If your fixture reacts to what you do, it is not measuring your change.
- As a filter for **choosing what to field**, determinism was never the property we needed. What we need is that the recording still makes sense against an opponent it never met.

The second is a matter of **degree**, and the degree is observable for free. A team's own episodes are a natural experiment, same agent, different opponents. What repeats across them is script; what is unique to one game is a response to that one opponent. Replayed elsewhere, the script still applies and the response is noise.

In [ ]:
from collections import Counter
import json

def canon(a):
    """Exact comparison, not approximate: sorted-key JSON."""
    return json.dumps(a, sort_keys=True, separators=(',', ':')) if a else '{}'

def scripted_fraction(stream, siblings):
    """Fraction of turns matching the team's OWN modal action at that turn."""
    matched = counted = 0
    for t in range(len(stream)):
        votes = Counter(canon(s[t]) for s in siblings if t < len(s))
        if not votes:
            continue
        counted += 1
        matched += votes.most_common(1)[0][0] == canon(stream[t])
    return matched / counted if counted else 0.0

# A scripted agent: same plan every game, one improvised turn.
plan     = [{'a': i % 4} for i in range(20)]
game_1   = plan[:7]  + [{'a': 99}] + plan[8:]
game_2   = plan[:13] + [{'a': 98}] + plan[14:]
game_3   = plan[:]

# A reactive agent: the plan is only a skeleton.
react_1  = [{'a': i % 4} if i % 3 == 0 else {'a': 50 + i} for i in range(20)]
react_2  = [{'a': i % 4} if i % 3 == 0 else {'a': 70 + i} for i in range(20)]
react_3  = [{'a': i % 4} if i % 3 == 0 else {'a': 90 + i} for i in range(20)]

print(f'scripted agent  {scripted_fraction(game_1,  [game_2, game_3]):.1%}')
print(f'reactive agent  {scripted_fraction(react_1, [react_2, react_3]):.1%}')

The binary test rejects *both* of those agents, because neither repeats byte-for-byte. The continuous one separates them, and it costs no games at all, only downloads.

**The honest caveat, and it matters.** This measures how *repetitive* a recording is. Calling that quantity 'script' is an interpretation, and the interpretation may be wrong even where the number is useful. A repetitive agent might be repetitive because the situations it meets are similar, not because it is following a plan. We have not separated those, and this notebook does not claim to.

## 4. Stage 2, the screen, and the way it quietly stops working

The screen replays each surviving candidate against a pool of real recorded opponents and counts wins. That worked until the day it silently stopped ranking anything.

One property of that screen is worth stating here rather than later: its eight opponents cluster into **four distinct behaviours**, so a 16 game result carries the weight of 8. Saturation was therefore not only a sign of a weak pool.

In [ ]:
import statistics

# Four candidates against a pool that is too weak for them.
# Wins saturate; the expected margin does not.
pool_results = {
    'candidate A': [5200, 4800, 6100, 5400, 5900, 5100, 4700, 6000],
    'candidate B': [2700, 2500, 3100, 2600, 2900, 2400, 2800, 2700],
    'candidate C': [ 300,  180,  420,  260,  350,  210,  290,  330],
    'incumbent  ': [5300, 5000, 5800, 5500, 5600, 5200, 5400, 5700],
}

print(f"{'':<13}{'wins':>7}{'mean margin':>14}")
for name, margins in pool_results.items():
    wins = sum(m > 0 for m in margins)
    print(f'{name:<13}{wins:>4}/{len(margins):<2}{statistics.mean(margins):>+14,.0f}')

print('\nEvery one wins every game. The win count cannot order them.')
print('The margin orders them, and it says C is nowhere near the incumbent.')

In [ ]:
# One chart in this notebook, so its style lives here with it: quiet axes,
# a light horizontal grid, colour by role (blue candidates, orange incumbent).
import matplotlib.pyplot as plt
import numpy as np
BLUE, ORANGE, GRID = '#2a78d6', '#eb6834', '#e7e7e4'
plt.rcParams.update({'figure.dpi': 130, 'font.size': 10,
                     'axes.spines.top': False, 'axes.spines.right': False})

def tidy(ax, ylab=None, title=None):
    ax.set_axisbelow(True)
    ax.yaxis.grid(True, color=GRID, lw=0.8)
    ax.xaxis.grid(False)
    if ylab: ax.set_ylabel(ylab)
    if title: ax.set_title(title, loc='left', pad=10)
    return ax

names = list(pool_results)
wins  = [sum(m > 0 for m in pool_results[k]) for k in names]
marg  = [statistics.mean(pool_results[k]) for k in names]
col   = [ORANGE if k.strip() == 'incumbent' else BLUE for k in names]

# Two panels, never two y-axes on one chart: the scales are unrelated.
fig, axes = plt.subplots(1, 2, figsize=(9.4, 3.5), sharey=False)
y = np.arange(len(names))[::-1]

ax = axes[0]
ax.barh(y, wins, color=col, height=0.6)
ax.set_yticks(y); ax.set_yticklabels([k.strip() for k in names], fontsize=9)
tidy(ax, None, 'Wins: nothing to rank')
ax.set_xlabel('games won of 8'); ax.set_xlim(0, 8.6)
ax.xaxis.grid(True, color=GRID, lw=0.8); ax.yaxis.grid(False)

ax = axes[1]
ax.barh(y, marg, color=col, height=0.6)
ax.set_yticks(y); ax.set_yticklabels([])
tidy(ax, None, 'Margin: an order appears')
ax.set_xlabel('mean margin')
ax.xaxis.grid(True, color=GRID, lw=0.8); ax.yaxis.grid(False)
plt.tight_layout(); plt.show()

**What this cost us before we noticed.** We ran a screen where our own fielded route and four candidates all won 16 of 16. Read as a win count, that says the candidates are as good as what we field. Read as a margin, one candidate was worth 2.5x the incumbent and two were worth half.

So the screen now reports **margin as well as wins**, and margin is also what we actually want to maximise: the expected difference between our bank and the opponent's.

**And a second, later lesson.** We added a head-to-head measure, candidate against the exact route it would replace, because it is exact when both are deterministic. For a while the two measures agreed, and we read the agreement as mutual confirmation. Then they disagreed: a candidate beat our fielded route head-to-head while performing at half its margin against the field.

That disagreement was more informative than the agreement had been. They measure different things, *beating one specific opponent* versus *beating the field*, and only the second is what the leaderboard pays for. The agreement had been luck, and we had been treating it as evidence.

## 5. The population IS the measurement

The screen's numbers only mean anything relative to who was in the pool, and this is the lesson that cost the most.

For a while candidates were ranked against a panel of the five **strongest** public agents available, because scoring against the hardest opposition felt like rigour. The ladder does not pay for beating the five best agents in the competition. It pays for beating whoever matchmaking serves, and those are mostly ordinary submissions. Re-scored against **344 recorded ladder routes** over 8,256 games each, the panel's ordering of five candidates did not soften. It **inverted**: the panel's favourite fell to third, and its near-last became first at 88.5 percent. The live ladder, read properly, had already agreed with the corpus before anyone asked.

The same experiment had a second axis, invisible until it was measured. With only **three episode seeds**, one fixed candidate's win rate against one fixed opponent set moved by up to 59 points across seeds, and paired candidates correlated at only r = +0.31, so common random numbers do not rescue a thin panel. The cell below shows what that does to a real effect.


In [ ]:
import math

# A real +13.7-point gap between two candidates, measured on k seeds, when one
# tape's win rate swings with a per-seed sd near 17 points and the candidates'
# swings barely correlate. These are the measured values from the experiment
# that inverted; only k varies.
sd_paired = 17.3
for k in (3, 6, 12, 24):
    half = 1.96 * sd_paired / math.sqrt(k)
    lo, hi = 13.7 - half, 13.7 + half
    verdict = 'spans zero: INVISIBLE' if lo < 0 else 'resolved'
    print(f'{k:>3} seeds: 95% CI on a true +13.7 gap = [{lo:+6.1f}, {hi:+6.1f}]  {verdict}')

print('\nTwelve seeds is the floor, and it is arithmetic, not taste.')


## 6. The judge, and the number that is the real sample size

Q1 sounds like it needs hundreds of games. It needs something scarcer: **discordant pairs**. Replay the candidate and the incumbent through the same real matchups, same rival, same seed, same seat, both LIVE, and every game they both win or both lose contributes nothing to the choice between them. The evidence lives entirely in the games where they disagree.

Two decisions from this project's log, shown exactly as the instrument printed them:


In [ ]:
from math import comb

def mcnemar_two_sided(a, b):
    n = a + b
    if n == 0:
        return 1.0
    k = max(a, b)
    return min(1.0, 2 * sum(comb(n, j) for j in range(k, n + 1)) / 2 ** n)

def pr_better(k, n):
    # P(Beta(k+1, n-k+1) > 1/2), closed binomial form
    a, b = k + 1, n - k + 1
    m = a + b - 1
    return sum(comb(m, i) for i in range(0, a)) / 2.0 ** m

cases = [
    ('candidate P vs incumbent', 106, 11, 5),
    ('candidate R vs incumbent', 106, 53, 0),
]
print(f'{"":<26}{"games":>6}{"discordant":>11}{"split":>8}{"p":>8}{"Pr[better]":>12}')
for name, games, up, down in cases:
    p = mcnemar_two_sided(up, down)
    pr = pr_better(up, up + down)
    print(f'{name:<26}{games:>6}{up+down:>11}{f"{up}-{down}":>8}{p:>8.3f}{pr:>11.1%}')

print()
print('Same game count. The first candidate was PROBABLY better and the slot')
print('stayed unspent, because 16 discordant pairs at p = 0.21 is a lean, not')
print('a result. The second was not a close call. The game count never knew')
print('the difference; the discordant count did.')


In [ ]:
from math import comb
import statistics

def sign_test(up, down):
    """Two-sided p on the games that actually moved."""
    n = up + down
    if n == 0:
        return 1.0
    k = max(up, down)
    return min(1.0, 2 * sum(comb(n, j) for j in range(k, n + 1)) / 2 ** n)

def report(label, margins):
    """Never the median alone: while behind, P(win) rises with dispersion."""
    s = sorted(margins)
    print(f'{label:<12} n {len(s):>3}   median {statistics.median(s):>+9,.0f}'
          f'   sigma {statistics.pstdev(s):>9,.0f}'
          f'   p90 {s[int(0.9 * (len(s) - 1))]:>+9,.0f}')

print('the screen result that looked convincing:')
print(f'  9 of 12 head-to-head   p = {sign_test(9, 3):.3f}  -> does NOT pass\n')
print('what would pass at the same win rate, with depth:')
print(f'  30 of 40 head-to-head  p = {sign_test(30, 10):.4f} -> passes\n')

# Two candidates with the SAME median and very different risk profiles.
steady   = [100] * 20
volatile = [-4000, 4200] * 10
report('steady', steady)
report('volatile', volatile)
print('\nIdentical means. If you are behind the bank you must beat,')
print('the volatile one is the better bet and the median hides that.')

### A worked example

The screen found a challenger ahead of our fielded route on both of its measures. The judge, at five times the depth, disagreed.

| | screen (8 opponents) | judge (40 opponents, paired) |
|---|---|---|
| challenger margin | **+5,640** | +6,260 |
| incumbent margin | +5,519 | +6,255 |
| the gap | **+121** | **+5** |
| wins | | 67/80 against 67/80, *identical* |
| paired sign test | | **p = 0.56, not significant** |

The screen's edge was sampling noise and vanished at depth. Head to head the challenger did win, 27 of 40, **p = 0.0385**, on a median margin of **+61 against banks near 100,000**. Statistically real, economically nothing.

**Then the two routes were compared directly: they agree on 94.9 % to 98.3 % of turns.** The challenger was a near copy of the agent it was challenging, which is the actual reason their performance against the field was indistinguishable.

Two things follow. A judge is worth having, since a screen alone would have spent a submission slot here. And a **dedupe step before evaluation is worth more**, because without one a 120 game evaluation can be spent comparing an agent to a near copy of itself, with nothing in its output able to reveal it. [The pool notebook](https://www.kaggle.com/code/destbreso/how-many-opponents-is-a-pool-of-forty) has that measurement.

### The power of the decision

Before running a comparison, ask what it could possibly see. A sample size is not a budget line, it is the resolution of the instrument, and an underpowered comparison does not return "no effect", it returns noise wearing the costume of a verdict.

The quantity to fix in advance is the smallest effect you would ACT on. Here that is about a 60 percent paired win rate: selecting the best of a screen and quoting its score inflates it by about 20 points here (24 to 27 in the worst measured cases), so a candidate screening at 65 confirms near 57, and anything whose true rate is below 60 is not worth a slot anyway. The cell below computes, exactly, the probability that a paired sign test at the usual threshold DETECTS a given true rate at each sample size. Two numbers from it are worth memorising: at 40 paired games a real 60 percent effect is noticed about one time in five, and the games are not the sample anyway, the discordant pairs are, which shrinks everything further.

The operational rule that follows: every measure declares the sample it needs, the run audits every measure against its own declaration, and a row that fails the audit prints as THIN with its minimum detectable effect at 80 percent power. **THIN means not enough evidence. It never means no effect, and it never carries into the next experiment.**


In [ ]:
from math import comb

def sign_test_power(p_true, n, alpha=0.05):
    # exact: find the two-sided rejection threshold at alpha under p=1/2,
    # then the probability of landing beyond it when the truth is p_true
    # (detection in the direction of the true effect).
    def p2(k, n):
        m = max(k, n - k)
        return min(1.0, 2 * sum(comb(n, j) for j in range(m, n + 1)) / 2 ** n)
    k_crit = next((k for k in range(n // 2, n + 1) if p2(k, n) <= alpha), None)
    if k_crit is None:
        return 0.0
    return sum(comb(n, k) * p_true ** k * (1 - p_true) ** (n - k)
               for k in range(k_crit, n + 1))

print(f'{"n paired":>9}{"detects a true 57.5%":>22}{"a true 60%":>12}{"a true 65%":>12}')
for n in (40, 106, 199, 352, 786):
    row = [sign_test_power(p, n) for p in (0.575, 0.60, 0.65)]
    print(f'{n:>9}' + ''.join(f'{x:>12.0%}' if i else f'{x:>22.0%}'
                              for i, x in enumerate(row)))

print()
print('And the minimum detectable effect at 80 percent power, the number a')
print('THIN row prints so "not significant" cannot be quietly read as "no')
print('effect":')
for n in (40, 106, 352):
    p = 0.5
    while sign_test_power(p, n) < 0.80:
        p += 0.001
    print(f'  n = {n:>4}: smallest true rate seen 80% of the time = {p:.1%}')


### When a replayed pool becomes ground truth

The judge's matchups are recordings, and a recording cannot react. Two things keep that honest.

First, the pool carries a built-in acceptance test: the agent that RECORDED those games must reproduce its own recorded banks when replayed through the harness. When the tooling assumed candidates were fixed tapes, a reactive incumbent failed that gate on 8 of 8 rows, and the gate was right: taping a reactive agent against an idle rival records a game it never plays. Rebuilt to run both agents live at the recorded seats and seeds, the incumbent reproduced **8 of 8 recorded ladder banks to the dollar**, which is the difference between simulating a pool and holding a piece of the actual ladder.

Second, the reactive lane exists separately: live-vs-live duels against adaptive public agents, **both seats always**, on seed panels chosen so the town's shop draws differ across games. For an agent that branches on the shop sequence, three fixed seeds would exercise three branches of sixty-four; the panel is drawn from a map of which seed produces which draw, so coverage is a choice instead of an accident.


## 7. The testbed: a simulator, three kinds of opponent, and what each one answers

None of the stages above are affordable without a place to play games, and the design of that place decides what the numbers can mean.

**The simulator.** Every game here runs in an engine-exact C++ port of the environment ([source on GitHub](https://github.com/destbreso/kaggriculture-cppsim); how it was built and validated is its own pair of notebooks, [kagsim, the engine at 2000 episodes per second](https://www.kaggle.com/code/destbreso/kagsim-the-engine-at-2000-episodes-per-second) and [From 1 to 24,000 episodes a second](https://www.kaggle.com/code/destbreso/from-1-to-24-000-episodes-a-second)), proven the only way that matters: recorded ladder episodes replay through it to the recorded banks, to the dollar, and in live mode the agent that recorded a pool reproduces its own eight sampled banks exactly. It has two speeds. **L0** replays tape against tape with no Python in the loop, at a fraction of a millisecond per full 720-turn game, which is what makes census-scale questions free. **L1** keeps YOUR policy live in Python, called every turn, against a recorded or live rival, at a few hundred milliseconds per game dominated entirely by your own code. The judge and every paired verdict in this notebook run at L1; the corpus scans run at L0. A third rung sits above both: a live duel where the RIVAL is also a thinking Python agent runs at seconds per game, roughly thirty to fifty L1 games' worth, which is why the reactive lane is an audit and not the workhorse. Without the port the whole ladder shifts up by orders of magnitude and the multi-hundred-game protocol stops being an afternoon tool.

**Three kinds of opponent, three different questions.** Treating them as interchangeable is how instruments lie.

* **Baselines** (an idle rival, a fixed reference script): they answer *does this agent function, and what is its plan worth against an empty book*. Cheap, deterministic, ideal for smoke tests and crash detection. What they must never do is SELECT: value alone and value in company point in opposite directions here, measured at its extreme by a good whose price is high all season alone and collapses to a fraction of that the moment a rival shares the book. An agent tuned against baselines is tuned for a market that does not exist.
* **Replays** (recorded rivals at their recorded seeds): the workhorse. They are real ladder behaviour, they are free to run at scale, and paired comparisons on them cancel the bias that is common to both candidates. Two disciplines keep them honest. First, classify a rival before pooling it: a recording of an ADAPTIVE agent is being scored on a game it never played, while the field's scripts replay faithfully. Second, remember the bound is a property of the FIELD: the interaction bias, the only part that can flip a paired verdict, measured under thirty units of margin against this field's barely-reactive donors, and that number expires the day the field gets more reactive.
* **Reactive agents, live** (executable rivals, both sides thinking): the control population. They answer the one question replays cannot: *what happens when the rival responds to me*. They are two orders of magnitude more expensive, so they are not the primary lane; they are the audit of it. The clearest thing one ever showed here: a tape that banks 150,914 alone on a world banked 43,929 against a co-seller on the same world, and the entire difference was four early purchases the rival's price pressure caused the engine to refuse in silence. No baseline and no replay could have produced that game.


### Both seats, and seeds as worlds

Two rules about coverage, both cheap and both violated by default.

**Both seats, always.** The engine is symmetric; a game is not. Seat 0 and seat 1 meet different boards from the same seed, and the requirements table below carries the measured price of forgetting this.

**A seed is not a random number, it is a WORLD.** The seed fixes the town: which shops unlock, in what order, at steps 72 and 144 for the first two. Eight shop types make 64 ordered pairs, and the strongest public agents in this competition branch on exactly that pair. So a duel on three fixed seeds does not test an agent three times; it tests three of its 64 worlds and says nothing about the other 61. The fix costs one afternoon: map seeds to their draw once, then choose evaluation panels for shop DIVERSITY instead of habit. The cell below is the arithmetic of why hoping does not work.


In [ ]:
from math import comb

# Expected number of DISTINCT first-two-shop pairs covered by n random seeds,
# under the uniform approximation (the real draw is measurably non-uniform,
# which only makes small panels worse: common pairs repeat, rare ones hide).
PAIRS = 64
def expected_cover(n):
    return PAIRS * (1 - (1 - 1 / PAIRS) ** n)

print(f'{"seeds":>6}{"expected pairs covered":>24}{"of 64":>8}')
for n in (3, 12, 64, 150, 300, 600):
    e = expected_cover(n)
    print(f'{n:>6}{e:>24.1f}{e/PAIRS:>8.0%}')

print()
print('Random seeds cover slowly and unevenly. A CHOSEN panel of 64 seeds,')
print('one per pair from a precomputed map, covers everything a 300-seed')
print('random panel almost covers, at a fifth of the cost. Mapping 600 seeds')
print('took minutes and found every one of the 64 pairs represented.')


### Capture, and the shelf life of everything above

The pools and corpora all come from the same place: downloaded replays of real episodes. Three practices make the difference between an archive and an instrument.

* **Capture your own games first.** Every episode your submissions play arrives with the resolved seed and both action streams inside the replay. That is a pool nobody can argue with: the rivals you were ACTUALLY paired with, at the seeds you actually met, with the recorded banks as the acceptance test the harness must reproduce before any candidate is scored on it.
* **Capture the leader continuously.** A reference corpus of the top team's episodes answers structural questions (is my land calendar late, is my herd shaped wrong, does my endgame leave tiles fallow) for the price of a diff, before any protocol run is spent.
* **And re-pull before you decide anything.** The field turns over with a measured half life of about TWO DAYS: clades sweep through, hold briefly, and go extinct within a week. A pool from a fortnight ago is a different population wearing the same filename, and every number computed against it inherits the staleness silently. The refresh costs minutes; the stale decision costs a submission slot.


## 8. The seed-variance law, or why screen gains evaporate

The newest scar, and the one most likely to save a reader money. Four separate improvement ideas this week followed the same arc: a clear gain at the seed where it was found, survival at a second seed, and death at the third.

The cleanest case was a table of counter-moves confirmed at TWO seeds per world: 46 of 64 worlds showed a gain somewhere, 37 cleared a screen threshold, 13 survived the second-seed confirm, and on a third seed those 13 went **3 wins, 11 losses, 12 ties**. Which move wins turned out to vary with the seed's own randomness, not with the world the table was keyed on. Fresh single-episode specialist tapes did the same: four passed a one-seed screen by +3,000 to +6,400 and all four lost the sibling-seed confirm, one by 44,000.


In [ ]:
# The funnel that ate itself: candidates surviving each stage of selection,
# then the out-of-selection seed. Screen and confirm both used for SELECTION
# leaves nothing held out; the third seed is the first honest reading.
stages = [('one-seed screen, gain > 0', 46),
          ('screen threshold, gain > 3k', 37),
          ('second-seed confirm', 13),
          ('third seed: still winning', 3)]
w = 46
for label, n in stages:
    bar = '#' * int(round(44 * n / w))
    print(f'{label:<30}{n:>4}  {bar}')

print()
print('The rule this argues for: a perturbation of a system validated on N')
print('seeds, selected on n << N seeds, keeps its screen gain only by luck.')
print('What survived, every time, was the change that is monotone by')
print('construction or has a provable loss floor, because those never needed')
print('the selection to be right.')


## 9. Stage 4, the live check, and the unit that fooled us

The last stage is the leaderboard, and it has one rule we learned the expensive way: **do not read a score before it converges, and do not judge convergence by eye.**

The subtle part is the unit. Ratings update in batches, so averaging drift over *wall-clock time* made a score moving 6-8 points per **episode** look like it was crawling at 9 points per **hour**. The unit is the episode, always. A convergence check that samples for thirteen minutes and sees zero new episodes has measured nothing, and must say so rather than report stability.

## 10. The requirements this imposes

Each is a property the evaluation must have, and each is easy to omit without the omission being visible in the output.

| requirement | what goes wrong without it |
|---|---|
| evaluate from **both seats** | an overlay scored **+510 and survived** on one seat; on both seats, all 28 games it changed flipped to losses |
| **wins and margin**, never bank alone | the market is shared; a change can raise your bank and raise the opponent's more |
| **screen and confirm on disjoint sets** | selecting the best of N and reporting its score inflated results here by 24-27 points |
| **dispersion, not just the median** | while behind, P(win) rises with variance and falls with the median |
| **declare the population** | a panel of the strongest agents inverted a five-candidate ranking against the real field |
| **twelve seeds or do not bother** | at three seeds the CI on a real +13.7-point effect spans zero |
| **count discordant pairs, not games** | 106 games can hold 16 pairs of evidence, or 53; the verdicts differ completely |
| **live mode for reactive agents** | taping one against an idle rival records a game it never plays; 8 of 8 alignment failures said so |
| **an identity arm that reads zero** | one arm equalled its own control by accident and printed a clean, meaningless number for a whole batch |
| **the unit is the episode** | wall-clock averaging made a fast-moving rating look settled |
| **verify the control is bit-identical** | new behaviour goes behind a flag defaulting off, compared action-by-action, or the comparison is not one |


## 11. Pros and cons, stated plainly

**What the instrument is good for**

- The free stage does real work. It reduces thousands of candidates to dozens without playing a game, which is the only reason the expensive stages are affordable at all.
- The stages measure genuinely different things, so their agreement is weak evidence rather than an echo, and their *disagreement* is informative.
- It is honest about ties. A screen that saturates now says so instead of returning an arbitrary order.
- Every stage is reproducible from cached inputs, so a conclusion can be re-audited later when the instrument itself falls under suspicion. It will.

**What it cannot do, and where it will mislead you**

- **The pool is not the leaderboard.** Every offline number is measured against opponents we chose. Our retracted instrument was internally consistent too.
- **It ranks, it does not explain.** It tells you which recording to prefer and nothing about why that recording is good. This is the limit we most want to remove and have not.
- **The filter's interpretation may be wrong.** It measures repetition; we call it script; those are not the same claim.
- **The evaluated set is not the candidate set.** When more candidates clear the threshold than the budget can play, some go unevaluated. The tool now prints how many, silent truncation reads exactly like 'we covered everything'.
- **Instruments expire.** Ours held 6 of the top 20 when written and 3 a day later. Re-pull before quoting.
- **Small n.** The stages were tuned on tens of measurements, not thousands. Several numbers here would not survive a proper confidence interval, and we try to say which.

## 12. What I actually learned

Four things, none of them about farming.

**A binary test can hide a continuous quantity, and rigour is a bad reason to keep it.** The determinism filter felt strict because it rejected 86 percent of candidates. It was rejecting them on a property nobody needed, and 'this filter is very selective' is not evidence that it selects for the right thing.

**Build the stages to disagree.** The most useful thing this instrument ever did was contradict itself: two measures that had agreed for weeks split apart, and the split revealed what each had been measuring all along. A funnel whose stages always agree has one stage.

**Most of the work is not measuring; it is noticing which quantity you measured.** Bank instead of margin. Team instead of submission. Minutes instead of episodes. One seat instead of two. A panel instead of the field. A control that was not the thing being improved. In every case the number was real, well computed, and answering a question adjacent to the one asked. No code review has ever caught one of these here; a repeated measurement usually has.

**Selection depth must not exceed validation depth.** Anything selected on fewer seeds than the system it perturbs was validated on will keep its gain only by luck. The changes that survive are the ones that never needed the selection to be right: monotone by construction, or carrying a proven loss floor.


---

*Written while the competition is still running; the specific selection results it refers to are held back until after it closes. The instrument is described here in full because the instrument is the part that generalises, and because writing down which parts of it are scars, rather than design, seemed more useful than another leaderboard number.*